In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
import ast
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import pandas as pd
import numpy as np
import timm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from tqdm.notebook import tqdm

BASE = "../input/competitions/birdclef-2026" if os.path.exists("../input/competitions/birdclef-2026") else "../input/birdclef-2026"
TRAIN_AUDIO_DIR = os.path.join(BASE, "train_audio")
SOUNDSCAPES_DIR = os.path.join(BASE, "train_soundscapes")

train_df = pd.read_csv(os.path.join(BASE, "train.csv"))
taxonomy_df = pd.read_csv(os.path.join(BASE, "taxonomy.csv"))
soundscapes_df = pd.read_csv(os.path.join(BASE, "train_soundscapes_labels.csv"))

CLASSES = taxonomy_df['primary_label'].unique().tolist()
NUM_CLASSES = len(CLASSES)
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

train_clean = pd.DataFrame({
    'filepath': train_df['filename'].apply(lambda x: os.path.join(TRAIN_AUDIO_DIR, x)),
    'primary_label': train_df['primary_label'],
    'all_labels': train_df.apply(lambda row: [row['primary_label']] + ast.literal_eval(row.get('secondary_labels', "[]")), axis=1),
    'is_soundscape': False,
    'end_time': -1
})

def time_to_seconds(t_str):
    parts = str(t_str).strip().split(':')
    if len(parts) == 3:
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(float(parts[2]))
    elif len(parts) == 2:
        return int(parts[0]) * 60 + int(float(parts[1]))
    else:
        return int(float(t_str))

def parse_soundscape_row(row):
    filename = str(row['filename'])
    if not filename.endswith('.ogg'):
        filename = f"{filename}.ogg"

    end_time = time_to_seconds(row['end'])
    primary_label = str(row['primary_label'])
    
    if primary_label == 'nocall':
        labels = []
    else:
        labels = [primary_label]
        
    return pd.Series({
        'filepath': os.path.join(SOUNDSCAPES_DIR, filename),
        'primary_label': primary_label,
        'all_labels': labels, 
        'is_soundscape': True,
        'end_time': end_time
    })

soundscapes_clean = soundscapes_df.apply(parse_soundscape_row, axis=1)

full_df = pd.concat([train_clean, soundscapes_clean], ignore_index=True)
full_df = full_df[full_df['primary_label'].isin(CLASSES) | (full_df['primary_label'] == 'nocall')].reset_index(drop=True)

print(f"Total samples: {len(full_df)} (Clean: {len(train_clean)}, Soundscapes: {len(soundscapes_clean)})")

Total samples: 35705 (Clean: 35549, Soundscapes: 1478)


In [3]:
class Config:
    SR = 32000
    DURATION = 5
    MAX_LENGTH = SR * DURATION
    N_MELS = 128
    N_FFT = 1024
    HOP_LENGTH = 512
    TARGET_FRAMES = 320
    
    BATCH_SIZE = 16
    NUM_WORKERS = 2
    LR = 1e-4
    EPOCHS = 10
    FOLDS = 5
    RUN_FOLD = 0
    
    MODEL_NAME = 'vit_small_patch16_224'

class_counts = full_df['primary_label'].value_counts()
rare_classes = class_counts[(class_counts < Config.FOLDS) & (class_counts.index != 'nocall')].index

dfs_to_add = []
for c in rare_classes:
    n_missing = Config.FOLDS - class_counts[c]
    class_df = full_df[full_df['primary_label'] == c]
    dfs_to_add.append(class_df.sample(n_missing, replace=True))

if dfs_to_add:
    full_df = pd.concat([full_df] + dfs_to_add, ignore_index=True)
    print(f"Додано {sum([len(df) for df in dfs_to_add])} дублікатів для рідкісних класів.")

skf = StratifiedKFold(n_splits=Config.FOLDS, shuffle=True, random_state=42)
full_df['fold'] = -1

valid_labels_mask = full_df['primary_label'] != 'nocall'
for fold, (_, val_idx) in enumerate(skf.split(full_df[valid_labels_mask], full_df[valid_labels_mask]['primary_label'])):
    full_df.loc[full_df.index[valid_labels_mask][val_idx], 'fold'] = fold

nocall_idx = full_df[full_df['primary_label'] == 'nocall'].index
for i, idx in enumerate(nocall_idx):
    full_df.loc[idx, 'fold'] = i % Config.FOLDS

train_df_fold = full_df[full_df['fold'] != Config.RUN_FOLD].reset_index(drop=True)
val_df_fold = full_df[full_df['fold'] == Config.RUN_FOLD].reset_index(drop=True)

print(f"Fold {Config.RUN_FOLD} -> Train: {len(train_df_fold)}, Val: {len(val_df_fold)}")

Додано 34 дублікатів для рідкісних класів.
Fold 0 -> Train: 28591, Val: 7148


In [4]:
class BirdCLEFDataset(Dataset):
    def __init__(self, df, config, is_train=True):
        self.df = df
        self.config = config
        self.is_train = is_train
        
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=config.SR, n_fft=config.N_FFT, hop_length=config.HOP_LENGTH,
            n_mels=config.N_MELS, f_min=50, f_max=14000
        )
        self.amp_to_db = torchaudio.transforms.AmplitudeToDB()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        if row['is_soundscape']:
            frame_offset = (row['end_time'] - self.config.DURATION) * self.config.SR
            num_frames = self.config.MAX_LENGTH
            try:
                waveform, sr = torchaudio.load(row['filepath'], frame_offset=frame_offset, num_frames=num_frames)
            except:
                waveform, sr = torchaudio.load(row['filepath'])
        else:
            waveform, sr = torchaudio.load(row['filepath'])
            
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        audio_len = waveform.shape[1]
        

        if audio_len > self.config.MAX_LENGTH:
            if row['is_soundscape']:
                start = 0
            elif self.is_train:
                start = np.random.randint(0, audio_len - self.config.MAX_LENGTH)
            else:
                start = (audio_len - self.config.MAX_LENGTH) // 2
            waveform = waveform[:, start:start + self.config.MAX_LENGTH]
        elif audio_len < self.config.MAX_LENGTH:
            pad_len = self.config.MAX_LENGTH - audio_len
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))

        mel_spec = self.amp_to_db(self.mel_transform(waveform))
        mel_spec = (mel_spec - mel_spec.min()) / (mel_spec.max() - mel_spec.min() + 1e-6)
        mel_spec = mel_spec * 2 - 1
        
        current_frames = mel_spec.shape[2]
        if current_frames < self.config.TARGET_FRAMES:
            mel_spec = F.pad(mel_spec, (0, self.config.TARGET_FRAMES - current_frames))
        elif current_frames > self.config.TARGET_FRAMES:
            mel_spec = mel_spec[:, :, :self.config.TARGET_FRAMES]
            
        mel_spec = mel_spec.expand(3, -1, -1)
        
        # Таргети
        target = torch.zeros(NUM_CLASSES, dtype=torch.float32)
        for label in row['all_labels']:
            if label in class_to_idx:
                target[class_to_idx[label]] = 1.0

        return mel_spec, target

train_loader = DataLoader(BirdCLEFDataset(train_df_fold, Config, is_train=True), 
                          batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=Config.NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(BirdCLEFDataset(val_df_fold, Config, is_train=False), 
                        batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS, pin_memory=True)

In [5]:
class BirdCLEFTransformerSED(nn.Module):
    def __init__(self, model_name=Config.MODEL_NAME, num_classes=NUM_CLASSES, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, dynamic_img_size=True, global_pool='')
        in_features = self.backbone.num_features
        self.fc1 = nn.Linear(in_features, in_features)
        self.fc_prob = nn.Linear(in_features, num_classes)
        self.fc_att = nn.Linear(in_features, num_classes)

    def forward(self, x):
        tokens = self.backbone(x)
        expected_patches = (Config.N_MELS // 16) * (Config.TARGET_FRAMES // 16)
        if tokens.shape[1] == expected_patches + 1:
            tokens = tokens[:, 1:, :] 
            
        x = torch.relu(self.fc1(tokens))
        framewise_probs = torch.sigmoid(self.fc_prob(x))
        framewise_att = torch.softmax(self.fc_att(x), dim=1)
        
        clipwise_probs = torch.sum(framewise_probs * framewise_att, dim=1) 
        clipwise_logits = torch.log(clipwise_probs / (1 - clipwise_probs + 1e-6) + 1e-6)
        return clipwise_logits

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BirdCLEFTransformerSED().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LR, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=Config.LR, steps_per_epoch=len(train_loader), epochs=Config.EPOCHS, pct_start=0.1
)

best_roc_auc = 0.0
save_path = f"best_transformer_fold{Config.RUN_FOLD}.pth"

print(f"Починаємо тренування на {device}...")

for epoch in range(Config.EPOCHS):
    model.train()
    train_loss = 0.0

    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS} [Train]")
    for images, targets in train_pbar:
        images, targets = images.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step()
        
        train_loss += loss.item()
        train_pbar.set_postfix(loss=loss.item())
        
    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0.0
    all_preds = []
    all_targets = []
    
    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS} [Val]")
    with torch.no_grad():
        for images, targets in val_pbar:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            val_loss += loss.item()

            probs = torch.sigmoid(outputs).cpu().numpy()
            all_preds.append(probs)
            all_targets.append(targets.cpu().numpy())
            
    avg_val_loss = val_loss / len(val_loader)
    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)

    try:
        valid_cols = np.where(all_targets.sum(axis=0) > 0)[0]
        val_roc_auc = roc_auc_score(all_targets[:, valid_cols], all_preds[:, valid_cols], average='macro')
    except ValueError:
        val_roc_auc = 0.0
        
    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val ROC-AUC: {val_roc_auc:.4f}")
    
    if val_roc_auc > best_roc_auc:
        print(f"Валідація покращилась ({best_roc_auc:.4f} --> {val_roc_auc:.4f}). Зберігаємо модель")
        best_roc_auc = val_roc_auc
        torch.save(model.state_dict(), save_path)

print(f"Тренування завершено. Найкращий ROC-AUC: {best_roc_auc:.4f}")
print(f"Ваги збережено у: {save_path}")

model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

Починаємо тренування на cuda...


Epoch 1/10 [Train]:   0%|          | 0/1787 [00:00<?, ?it/s]

Epoch 1/10 [Val]:   0%|          | 0/447 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 0.1008 | Val Loss: 0.0285 | Val ROC-AUC: 0.6510
Валідація покращилась (0.0000 --> 0.6510). Зберігаємо модель


...

Epoch 9/10 [Val]:   0%|          | 0/447 [00:00<?, ?it/s]

Epoch 9 | Train Loss: 0.0087 | Val Loss: 0.0118 | Val ROC-AUC: 0.9587
Валідація покращилась (0.9582 --> 0.9587). Зберігаємо модель


Epoch 10/10 [Train]:   0%|          | 0/1787 [00:00<?, ?it/s]

Epoch 10/10 [Val]:   0%|          | 0/447 [00:00<?, ?it/s]

Epoch 10 | Train Loss: 0.0084 | Val Loss: 0.0118 | Val ROC-AUC: 0.9587
Валідація покращилась (0.9587 --> 0.9587). Зберігаємо модель
Тренування завершено. Найкращий ROC-AUC: 0.9587
Ваги збережено у: best_transformer_fold0.pth
